In [1]:
# pip install sentence-transformers torch

  Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 5.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 5.3 MB/s  0:00:02 eta 0:00:01
Using cached tokenizers-0.22.2-cp39-a

In [2]:
import numpy as np
import json
from typing import List, Dict, Tuple
from dataclasses import dataclass, asdict
import time
import re
from datetime import datetime
 
from sentence_transformers import SentenceTransformer

/Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
@dataclass
class TextChunk:
    """Represents a chunk of transcript with metadata."""
    
    chunk_id: int
    video_id: str
    source: str  # 'youtube' or 'instagram'
    text: str
    timestamp_start: float = 0.0  # seconds
    timestamp_end: float = 0.0
    chunk_size: int = 0
    
    def __post_init__(self):
        self.chunk_size = len(self.text.split())
 
class TranscriptChunker:
    """Service for chunking transcripts intelligently."""
    
    def __init__(self, chunk_size: int = 256, overlap: int = 50):
        """
        Initialize chunker.
        
        Args:
            chunk_size: Target words per chunk
            overlap: Words to overlap between chunks
        """
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def chunk_simple(self, text: str, video_id: str, source: str) -> List[TextChunk]:
        """
        Simple chunking by word count.
        
        Split text into chunks of ~chunk_size words with overlap.
        """
        
        words = text.split()
        chunks = []
        chunk_id = 0
        
        for i in range(0, len(words), self.chunk_size - self.overlap):
            chunk_words = words[i:i + self.chunk_size]
            
            if len(chunk_words) == 0:
                continue
            
            chunk_text = ' '.join(chunk_words)
            
            chunks.append(TextChunk(
                chunk_id=chunk_id,
                video_id=video_id,
                source=source,
                text=chunk_text
            ))
            
            chunk_id += 1
        
        return chunks
    
    def chunk_by_sentences(self, text: str, video_id: str, source: str) -> List[TextChunk]:
        """
        Chunk by sentences, keeping related sentences together.
        
        More semantic than word-based chunking.
        """
        
        # Split by common sentence endings
        sentences = re.split(r'(?<=[.!?])\s+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        chunks = []
        chunk_id = 0
        current_chunk = []
        current_word_count = 0
        
        for sentence in sentences:
            words_in_sentence = len(sentence.split())
            
            # Check if adding this sentence exceeds chunk_size
            if current_word_count + words_in_sentence > self.chunk_size and current_chunk:
                # Save current chunk
                chunk_text = ' '.join(current_chunk)
                chunks.append(TextChunk(
                    chunk_id=chunk_id,
                    video_id=video_id,
                    source=source,
                    text=chunk_text
                ))
                
                # Start new chunk with overlap
                chunk_id += 1
                
                # Keep last few sentences for overlap
                overlap_sentences = self._get_overlap_sentences(current_chunk, self.overlap)
                current_chunk = overlap_sentences + [sentence]
                current_word_count = sum(len(s.split()) for s in current_chunk)
            else:
                current_chunk.append(sentence)
                current_word_count += words_in_sentence
        
        # Add final chunk
        if current_chunk:
            chunk_text = ' '.join(current_chunk)
            chunks.append(TextChunk(
                chunk_id=chunk_id,
                video_id=video_id,
                source=source,
                text=chunk_text
            ))
        
        return chunks
    
    def chunk_with_timestamps(self, 
                             text: str, 
                             video_id: str, 
                             source: str,
                             transcript_entries: List[Dict]) -> List[TextChunk]:
        """
        Chunk by word count while preserving timestamp information.
        
        Useful for linking chunks back to video segments.
        """
        
        chunks = []
        chunk_id = 0
        
        # Create word-to-timestamp mapping
        word_timestamps = self._create_word_timestamp_mapping(transcript_entries)
        
        words = text.split()
        
        for i in range(0, len(words), self.chunk_size - self.overlap):
            chunk_words = words[i:i + self.chunk_size]
            
            if len(chunk_words) == 0:
                continue
            
            chunk_text = ' '.join(chunk_words)
            
            # Get timestamps for this chunk
            start_idx = i
            end_idx = min(i + self.chunk_size, len(words))
            
            timestamp_start = word_timestamps.get(start_idx, 0.0)
            timestamp_end = word_timestamps.get(end_idx - 1, 0.0)
            
            chunk = TextChunk(
                chunk_id=chunk_id,
                video_id=video_id,
                source=source,
                text=chunk_text,
                timestamp_start=timestamp_start,
                timestamp_end=timestamp_end
            )
            
            chunks.append(chunk)
            chunk_id += 1
        
        return chunks
    
    def _create_word_timestamp_mapping(self, transcript_entries: List[Dict]) -> Dict[int, float]:
        """Create mapping from word index to timestamp."""
        
        mapping = {}
        word_count = 0
        
        for entry in transcript_entries:
            text = entry.get('text', '')
            timestamp = entry.get('start', 0.0)
            
            words = text.split()
            for _ in words:
                mapping[word_count] = timestamp
                word_count += 1
        
        return mapping
    
    def _get_overlap_sentences(self, sentences: List[str], overlap_words: int) -> List[str]:
        """Get last N sentences that total ~overlap_words."""
        
        total = 0
        overlap_sentences = []
        
        for sentence in reversed(sentences):
            word_count = len(sentence.split())
            if total + word_count <= overlap_words:
                overlap_sentences.insert(0, sentence)
                total += word_count
            else:
                break
        
        return overlap_sentences
 
# Test chunking
print("=" * 80)
print("TEST 1: Transcript Chunking")
print("=" * 80)
 
sample_transcript = """
The future of artificial intelligence is here. Machine learning models can now understand
and generate human-like text. This revolution started with transformers. Transformers
changed everything. They use attention mechanisms. Attention allows models to focus on
important parts. The architecture is elegant. Self-attention compares each word to every
other word. This creates powerful representations. Large language models emerged. These
models are trained on billions of parameters. They can perform many tasks. Few-shot learning
became possible. Models understand context better. This enables better performance. The
applications are endless. From chatbots to code generation. From image understanding to
multimodal learning. AI is transforming industries. Healthcare benefits from AI. Finance
uses AI for predictions. Autonomous vehicles need AI. Education is being revolutionized.
The impact will be profound.
"""
 
chunker = TranscriptChunker(chunk_size=50, overlap=10)
 
# Test different chunking strategies
print("\n1. Word-based chunking (chunk_size=50, overlap=10)")
chunks_word = chunker.chunk_simple(sample_transcript, 'test_video_1', 'youtube')
print(f"   Generated {len(chunks_word)} chunks")
for chunk in chunks_word[:2]:
    print(f"   Chunk {chunk.chunk_id}: {chunk.text[:80]}... ({chunk.chunk_size} words)")
 
print("\n2. Sentence-based chunking")
chunks_sentence = chunker.chunk_by_sentences(sample_transcript, 'test_video_1', 'youtube')
print(f"   Generated {len(chunks_sentence)} chunks")
for chunk in chunks_sentence[:2]:
    print(f"   Chunk {chunk.chunk_id}: {chunk.text[:80]}... ({chunk.chunk_size} words)")
 
# Test with timestamps
print("\n3. Chunking with timestamps")
sample_entries = [
    {'text': 'The future of artificial intelligence is here', 'start': 0.0, 'duration': 5},
    {'text': 'Machine learning models can now understand and generate human-like text', 'start': 5.0, 'duration': 6},
    {'text': 'This revolution started with transformers', 'start': 11.0, 'duration': 4},
]
 
# Reconstruct transcript from entries
transcript_with_ts = ' '.join([e['text'] for e in sample_entries])
chunks_ts = chunker.chunk_with_timestamps(
    transcript_with_ts, 
    'test_video_2', 
    'youtube',
    sample_entries
)
print(f"   Generated {len(chunks_ts)} chunks with timestamps")
for chunk in chunks_ts:
    print(f"   Chunk {chunk.chunk_id}: {chunk.timestamp_start:.1f}s-{chunk.timestamp_end:.1f}s | {chunk.text[:60]}...")

TEST 1: Transcript Chunking

1. Word-based chunking (chunk_size=50, overlap=10)
   Generated 4 chunks
   Chunk 0: The future of artificial intelligence is here. Machine learning models can now u... (50 words)
   Chunk 1: elegant. Self-attention compares each word to every other word. This creates pow... (50 words)

2. Sentence-based chunking
   Generated 3 chunks
   Chunk 0: The future of artificial intelligence is here. Machine learning models can now u... (49 words)
   Chunk 1: Self-attention compares each word to every
other word. This creates powerful rep... (50 words)

3. Chunking with timestamps
   Generated 1 chunks with timestamps
   Chunk 0: 0.0s-11.0s | The future of artificial intelligence is here Machine learni...


In [5]:
class EmbeddingService:
    """Service for generating embeddings locally using SentenceTransformers."""
    
    def __init__(self, model_name: str = 'BAAI/bge-small-en-v1.5'):
        """
        Initialize embedding service.
        
        Args:
            model_name: HuggingFace model identifier
        """
        
        print(f"Loading embedding model: {model_name}")
        self.model = SentenceTransformer(model_name)
        self.model_name = model_name
        if hasattr(self.model, "get_embedding_dimension"):
            self.embedding_dimension = self.model.get_embedding_dimension()
        else:
            self.embedding_dimension = self.model.get_sentence_embedding_dimension()
        
        print(f"✅ Model loaded successfully")
        print(f"   Dimension: {self.embedding_dimension}")
        print(f"   Max length: {self.model.max_seq_length}")
    
    def embed_text(self, text: str, normalize: bool = True, as_tensor: bool = False) -> np.ndarray:
        """
        Generate embedding for a single text.
        
        Args:
            text: Text to embed
            normalize: Whether to normalize the embedding (recommended)
            as_tensor: Whether to return a torch Tensor
            
        Returns:
            Numpy array or torch Tensor of shape (embedding_dimension,)
        """
        
        embedding = self.model.encode(
            text,
            normalize_embeddings=normalize,
            convert_to_tensor=as_tensor
        )
        return embedding
    
    def embed_batch(
        self,
        texts: List[str],
        normalize: bool = True,
        batch_size: int = 32,
        as_tensor: bool = False
    ) -> List[np.ndarray]:
        """
        Generate embeddings for multiple texts efficiently.
        
        Args:
            texts: List of texts to embed
            normalize: Whether to normalize embeddings
            batch_size: Number of texts to process at once
            as_tensor: Whether to return a torch Tensor
            
        Returns:
            List of numpy arrays or a torch Tensor
        """
        
        embeddings = self.model.encode(
            texts,
            normalize_embeddings=normalize,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_tensor=as_tensor
        )
        
        if as_tensor:
            return embeddings
        return [embedding for embedding in embeddings]
    
    def semantic_search(self, query: str, corpus: List[str], top_k: int = 5) -> List[Dict]:
        """
        Find most similar texts to a query.
        
        Args:
            query: Query text
            corpus: List of texts to search
            top_k: Number of results to return
            
        Returns:
            List of dicts with corpus_id, score, text
        """
        
        from sentence_transformers.util import semantic_search as ss
        
        query_embedding = self.embed_text(query, as_tensor=True)
        corpus_embeddings = self.embed_batch(corpus, as_tensor=True)
        
        results = ss(query_embedding, corpus_embeddings, top_k=top_k)
        
        # Format results
        formatted = []
        for result in results[0]:  # ss returns list of lists
            formatted.append({
                'corpus_id': result['corpus_id'],
                'score': float(result['score']),
                'text': corpus[result['corpus_id']]
            })
        
        return formatted
    
    def calculate_similarity(self, text1: str, text2: str) -> float:
        """Calculate cosine similarity between two texts."""
        
        from sentence_transformers.util import cos_sim
        
        emb1 = self.embed_text(text1, as_tensor=True)
        emb2 = self.embed_text(text2, as_tensor=True)
        
        similarity = cos_sim(emb1, emb2)
        return float(similarity[0][0])
 
# Test embeddings (requires model download on first run)
print("\n" + "=" * 80)
print("TEST 2: Embedding Generation")
print("=" * 80)
 
try:
    embedding_service = EmbeddingService()
    
    # Test single embedding
    print("\n1. Single text embedding")
    test_text = "Artificial intelligence is transforming the world"
    embedding = embedding_service.embed_text(test_text)
    print(f"   Text: '{test_text}'")
    print(f"   Embedding shape: {embedding.shape}")
    print(f"   First 5 dimensions: {embedding[:5]}")
    
    # Test batch embedding
    print("\n2. Batch embeddings")
    batch_texts = [
        "Machine learning is a subset of AI",
        "Deep learning uses neural networks",
        "The weather is nice today"
    ]
    batch_embeddings = embedding_service.embed_batch(batch_texts)
    print(f"   Processed {len(batch_texts)} texts")
    print(f"   Embeddings generated: {len(batch_embeddings)}")
    
    # Test semantic search
    print("\n3. Semantic search")
    query = "What is artificial intelligence?"
    corpus = batch_texts
    results = embedding_service.semantic_search(query, corpus, top_k=3)
    print(f"   Query: '{query}'")
    print(f"   Results:")
    for result in results:
        print(f"      {result['score']:.4f}: {result['text']}")
    
    # Test similarity
    print("\n4. Text similarity")
    text_a = "Machine learning models are powerful"
    text_b = "AI models are very capable"
    text_c = "The sky is blue"
    
    sim_ab = embedding_service.calculate_similarity(text_a, text_b)
    sim_ac = embedding_service.calculate_similarity(text_a, text_c)
    
    print(f"   Text A: '{text_a}'")
    print(f"   Text B: '{text_b}'")
    print(f"   Text C: '{text_c}'")
    print(f"   Similarity A-B: {sim_ab:.4f} (related)")
    print(f"   Similarity A-C: {sim_ac:.4f} (unrelated)")
    
except Exception as e:
    print(f"⚠️ Embedding service error: {e}")
    print("   Make sure torch and sentence-transformers are installed")


TEST 2: Embedding Generation
Loading embedding model: BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7373.31it/s]


✅ Model loaded successfully
   Dimension: 384
   Max length: 512

1. Single text embedding
   Text: 'Artificial intelligence is transforming the world'
   Embedding shape: (384,)
   First 5 dimensions: [-0.00416927 -0.00064791  0.0530661  -0.02801901  0.02314163]

2. Batch embeddings


Batches: 100%|██████████| 1/1 [00:00<00:00, 86.68it/s]


   Processed 3 texts
   Embeddings generated: 3

3. Semantic search


Batches: 100%|██████████| 1/1 [00:00<00:00, 93.40it/s]


   Query: 'What is artificial intelligence?'
   Results:
      0.7517: Machine learning is a subset of AI
      0.5866: Deep learning uses neural networks
      0.4078: The weather is nice today

4. Text similarity
   Text A: 'Machine learning models are powerful'
   Text B: 'AI models are very capable'
   Text C: 'The sky is blue'
   Similarity A-B: 0.8294 (related)
   Similarity A-C: 0.4603 (unrelated)


In [6]:
@dataclass
class EmbeddedChunk:
    """Chunk with generated embedding."""
    
    chunk: TextChunk
    embedding: np.ndarray
    embedding_model: str
    embedding_dimension: int
    generated_at: str
    
    def to_dict_for_storage(self) -> Dict:
        """Convert to dict for storing in vector DB."""
        
        return {
            'chunk_id': self.chunk.chunk_id,
            'video_id': self.chunk.video_id,
            'source': self.chunk.source,
            'text': self.chunk.text,
            'timestamp_start': self.chunk.timestamp_start,
            'timestamp_end': self.chunk.timestamp_end,
            'embedding': self.embedding.tolist(),  # Vector databases expect lists
            'embedding_model': self.embedding_model,
            'embedding_dimension': self.embedding_dimension,
        }
 
class ChunkEmbeddingPipeline:
    """Pipeline for chunking and embedding transcripts."""
    
    def __init__(self, 
                 chunk_size: int = 256,
                 overlap: int = 50,
                 embedding_model: str = 'BAAI/bge-small-en-v1.5'):
        """
        Initialize pipeline.
        
        Args:
            chunk_size: Words per chunk
            overlap: Word overlap between chunks
            embedding_model: Model for embeddings
        """
        
        self.chunker = TranscriptChunker(chunk_size=chunk_size, overlap=overlap)
        self.embedding_service = EmbeddingService(model_name=embedding_model)
    
    def process_transcript(self,
                          transcript: str,
                          video_id: str,
                          source: str,
                          transcript_entries: List[Dict] = None) -> Tuple[List[EmbeddedChunk], Dict]:
        """
        Complete pipeline: chunk transcript and generate embeddings.
        
        Returns:
            Tuple of (embedded_chunks, performance_metrics)
        """
        
        metrics = {
            'video_id': video_id,
            'source': source,
            'start_time': datetime.now().isoformat(),
            'chunks_created': 0,
            'embeddings_generated': 0,
            'total_time_seconds': 0.0,
            'chunking_time_seconds': 0.0,
            'embedding_time_seconds': 0.0,
        }
        
        start_total = time.time()
        
        # Step 1: Chunk
        start_chunk = time.time()
        
        if transcript_entries:
            chunks = self.chunker.chunk_with_timestamps(
                transcript, video_id, source, transcript_entries
            )
        else:
            chunks = self.chunker.chunk_by_sentences(transcript, video_id, source)
        
        metrics['chunking_time_seconds'] = time.time() - start_chunk
        metrics['chunks_created'] = len(chunks)
        
        print(f"✅ Created {len(chunks)} chunks in {metrics['chunking_time_seconds']:.2f}s")
        
        # Step 2: Embed all chunks
        start_embed = time.time()
        
        chunk_texts = [chunk.text for chunk in chunks]
        embeddings = self.embedding_service.embed_batch(chunk_texts)
        
        metrics['embedding_time_seconds'] = time.time() - start_embed
        metrics['embeddings_generated'] = len(embeddings)
        
        print(f"✅ Generated {len(embeddings)} embeddings in {metrics['embedding_time_seconds']:.2f}s")
        
        # Step 3: Pair chunks with embeddings
        embedded_chunks = []
        
        for chunk, embedding in zip(chunks, embeddings):
            embedded_chunk = EmbeddedChunk(
                chunk=chunk,
                embedding=embedding,
                embedding_model=self.embedding_service.model_name,
                embedding_dimension=self.embedding_service.embedding_dimension,
                generated_at=datetime.now().isoformat()
            )
            embedded_chunks.append(embedded_chunk)
        
        metrics['total_time_seconds'] = time.time() - start_total
        
        return embedded_chunks, metrics
 
# Test pipeline
print("\n" + "=" * 80)
print("TEST 3: Complete Chunking & Embedding Pipeline")
print("=" * 80)
 
try:
    pipeline = ChunkEmbeddingPipeline(chunk_size=100, overlap=20)
    
    # Process sample transcript
    sample_transcript = """
    Artificial intelligence is fundamentally changing how we work and live.
    Machine learning algorithms learn from data to improve performance.
    Deep learning uses neural networks with multiple layers.
    Transformers revolutionized natural language processing.
    Attention mechanisms allow models to focus on important information.
    Large language models like GPT can generate human-like text.
    These models are trained on massive datasets.
    Transfer learning lets us use pre-trained models for new tasks.
    Fine-tuning adapts models to specific domains.
    Reinforcement learning enables models to learn from interaction.
    """
    
    embedded_chunks, metrics = pipeline.process_transcript(
        transcript=sample_transcript,
        video_id='test_video_001',
        source='youtube'
    )
    
    print(f"\n📊 Pipeline Metrics:")
    print(f"   Total time: {metrics['total_time_seconds']:.2f}s")
    print(f"   Chunking time: {metrics['chunking_time_seconds']:.2f}s")
    print(f"   Embedding time: {metrics['embedding_time_seconds']:.2f}s")
    print(f"   Chunks: {metrics['chunks_created']}")
    print(f"   Embeddings: {metrics['embeddings_generated']}")
    
    if metrics['chunks_created'] > 0:
        avg_time_per_chunk = (metrics['chunking_time_seconds'] / metrics['chunks_created']) * 1000
        print(f"   Time per chunk: {avg_time_per_chunk:.2f}ms")
    
    if metrics['embeddings_generated'] > 0:
        avg_time_per_embedding = (metrics['embedding_time_seconds'] / metrics['embeddings_generated']) * 1000
        print(f"   Time per embedding: {avg_time_per_embedding:.2f}ms")
    
    # Display sample embedded chunks
    print(f"\n📝 Sample Embedded Chunks:")
    for i, emb_chunk in enumerate(embedded_chunks[:2]):
        print(f"\n   Chunk {i}:")
        print(f"      Text: {emb_chunk.chunk.text[:80]}...")
        print(f"      Embedding shape: {emb_chunk.embedding.shape}")
        print(f"      First 3 dimensions: {emb_chunk.embedding[:3]}")
 
except Exception as e:
    print(f"⚠️ Pipeline error: {e}")


TEST 3: Complete Chunking & Embedding Pipeline
Loading embedding model: BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8081.20it/s]


✅ Model loaded successfully
   Dimension: 384
   Max length: 512
✅ Created 1 chunks in 0.00s


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.80it/s]

✅ Generated 1 embeddings in 0.27s

📊 Pipeline Metrics:
   Total time: 0.27s
   Chunking time: 0.00s
   Embedding time: 0.27s
   Chunks: 1
   Embeddings: 1
   Time per chunk: 0.20ms
   Time per embedding: 267.82ms

📝 Sample Embedded Chunks:

   Chunk 0:
      Text: Artificial intelligence is fundamentally changing how we work and live. Machine ...
      Embedding shape: (384,)
      First 3 dimensions: [-0.00383996  0.03268147  0.02289238]


In [7]:
def compare_embedding_models():
    """
    Compare different embedding models.
    Tests speed, quality, and suitability.
    """
    
    models_to_test = [
        'BAAI/bge-small-en-v1.5',      # Small, fast, good quality
        'intfloat/e5-small-v2',         # Lightweight, versatile
        'sentence-transformers/all-MiniLM-L6-v2',  # Very small
    ]
    
    test_text = "Artificial intelligence and machine learning are transforming industries"
    
    print("\n" + "=" * 80)
    print("COMPARISON: Embedding Models")
    print("=" * 80)
    
    results = []
    
    for model_name in models_to_test:
        try:
            print(f"\nTesting: {model_name}")
            
            start = time.time()
            service = EmbeddingService(model_name=model_name)
            load_time = time.time() - start
            
            start = time.time()
            embedding = service.embed_text(test_text)
            embed_time = time.time() - start
            
            result = {
                'model': model_name,
                'dimension': service.embedding_dimension,
                'load_time': load_time,
                'embed_time': embed_time,
                'size_category': 'small' if service.embedding_dimension <= 384 else 'medium'
            }
            
            results.append(result)
            
            print(f"   ✅ Dimension: {result['dimension']}")
            print(f"   Load time: {result['load_time']:.2f}s")
            print(f"   Embed time: {result['embed_time']:.4f}s")
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
    
    # Summary table
    print("\n" + "=" * 80)
    print("MODEL COMPARISON SUMMARY")
    print("=" * 80)
    
    print(f"\n{'Model':<45} {'Dim':>5} {'Load':>8} {'Embed':>8}")
    print("-" * 80)
    
    for r in results:
        model_short = r['model'].split('/')[-1][:40]
        print(f"{model_short:<45} {r['dimension']:>5} {r['load_time']:>7.2f}s {r['embed_time']:>7.4f}s")
    
    return results

In [8]:
compare_embedding_models()


COMPARISON: Embedding Models

Testing: BAAI/bge-small-en-v1.5
Loading embedding model: BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7313.81it/s]


✅ Model loaded successfully
   Dimension: 384
   Max length: 512
   ✅ Dimension: 384
   Load time: 5.79s
   Embed time: 0.2629s

Testing: intfloat/e5-small-v2
Loading embedding model: intfloat/e5-small-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 26427.71it/s]


✅ Model loaded successfully
   Dimension: 384
   Max length: 512
   ✅ Dimension: 384
   Load time: 32.41s
   Embed time: 0.1449s

Testing: sentence-transformers/all-MiniLM-L6-v2
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7169.27it/s]


✅ Model loaded successfully
   Dimension: 384
   Max length: 256
   ✅ Dimension: 384
   Load time: 5.51s
   Embed time: 0.0118s

MODEL COMPARISON SUMMARY

Model                                           Dim     Load    Embed
--------------------------------------------------------------------------------
bge-small-en-v1.5                               384    5.79s  0.2629s
e5-small-v2                                     384   32.41s  0.1449s
all-MiniLM-L6-v2                                384    5.51s  0.0118s


[{'model': 'BAAI/bge-small-en-v1.5',
  'dimension': 384,
  'load_time': 5.793594121932983,
  'embed_time': 0.2629110813140869,
  'size_category': 'small'},
 {'model': 'intfloat/e5-small-v2',
  'dimension': 384,
  'load_time': 32.40544319152832,
  'embed_time': 0.14490199089050293,
  'size_category': 'small'},
 {'model': 'sentence-transformers/all-MiniLM-L6-v2',
  'dimension': 384,
  'load_time': 5.51396107673645,
  'embed_time': 0.011753082275390625,
  'size_category': 'small'}]

In [9]:
def evaluate_chunking_strategies(transcript: str, video_id: str):
    """
    Compare different chunking strategies.
    Evaluates coverage, redundancy, and semantic coherence.
    """
    
    print("\n" + "=" * 80)
    print("EVALUATION: Chunking Strategies")
    print("=" * 80)
    
    chunker = TranscriptChunker()
    
    strategies = {
        'word_based': chunker.chunk_simple,
        'sentence_based': chunker.chunk_by_sentences,
    }
    
    results = {}
    
    for name, strategy in strategies.items():
        print(f"\n{name.upper().replace('_', ' ')}:")
        
        chunks = strategy(transcript, video_id, 'youtube')
        
        # Calculate metrics
        total_words = len(transcript.split())
        total_chunk_words = sum(c.chunk_size for c in chunks)
        overlap_ratio = (total_chunk_words - total_words) / total_words * 100
        
        print(f"   Chunks: {len(chunks)}")
        print(f"   Total words in transcript: {total_words}")
        print(f"   Total words in chunks: {total_chunk_words}")
        print(f"   Overlap: {overlap_ratio:.1f}%")
        print(f"   Avg chunk size: {total_chunk_words / len(chunks):.0f} words")
        
        results[name] = {
            'num_chunks': len(chunks),
            'overlap_ratio': overlap_ratio,
            'avg_chunk_size': total_chunk_words / len(chunks) if chunks else 0
        }
    
    return results
 
# Run evaluation
sample_long_transcript = " ".join([sample_transcript] * 5)  # Make it longer
evaluate_chunking_strategies(sample_long_transcript, 'test_video')


EVALUATION: Chunking Strategies

WORD BASED:
   Chunks: 2
   Total words in transcript: 405
   Total words in chunks: 455
   Overlap: 12.3%
   Avg chunk size: 228 words

SENTENCE BASED:
   Chunks: 2
   Total words in transcript: 405
   Total words in chunks: 455
   Overlap: 12.3%
   Avg chunk size: 228 words


{'word_based': {'num_chunks': 2,
  'overlap_ratio': 12.345679012345679,
  'avg_chunk_size': 227.5},
 'sentence_based': {'num_chunks': 2,
  'overlap_ratio': 12.345679012345679,
  'avg_chunk_size': 227.5}}